# Credit Analyst Agent - Asaan 

**Assignment 1:** Fine-tuning the loan-risk classifier (v6) -
final checkpoint achieves 93.33% accuracy, balanced across approve/decline/refer.

**Assignment 2:** Wraps finetuned model in an agent for a credit-analyst 
use case: reviewing loan applications, calculating DTI/DSCR/LTV, pulling 
mock credit bureau data, and producing a structured approve/decline/refer 
memo. Built as a deterministic LangChain (LCEL) pipeline with a policy-rule 
gate, a separate deterministic rule for commercial/DSCR-driven loans (model 
was only fine-tuned on retail profiles), and finetuned model itself for retail judgment.

### 1. Setup: install libraries

In [23]:
!pip install -q unsloth
!pip install -q langchain langchain-core

### 2. Load model + tokenizer (Qwen2.5-1.5B-Instruct, 4-bit)

In [5]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    max_seq_length = 1024,
    dtype = None,
    load_in_4bit = True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

### 3. Format synthetic training data into chat template

In [6]:
import json

def format_example(entry):
    profile = entry["profile"]
    label = entry["label"]
    reasoning = entry["reasoning"]

    user_msg = f"Classify the following loan application as approve, decline, or refer, and explain your reasoning.\n\nApplication: {profile}"
    assistant_msg = f"{label.upper()}: {reasoning}"

    messages = [
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": assistant_msg},
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}
    
# Load your training data (update path to match your Kaggle dataset)
with open("/kaggle/input/datasets/ajawad06/loan-risk-data/train.jsonl", "r") as f:
    data = [json.loads(line) for line in f]

# Format each example
formatted = [format_example(e) for e in data]

# Save as a new jsonl ready for the Trainer
with open("/kaggle/working/train_formatted.jsonl", "w") as f:
    for item in formatted:
        f.write(json.dumps(item) + "\n")

print(f"Formatted {len(formatted)} examples")
print("\nExample output:\n")
print(formatted[0]["text"])

Formatted 324 examples

Example output:

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Classify the following loan application as approve, decline, or refer, and explain your reasoning.

Application: Applicant earns $81,000 annually, credit score 758, requests a $48,000 solar installation loan. Existing mortgage is affordable, payment history perfect, employed with the same utility company for 13 years.<|im_end|>
<|im_start|>assistant
APPROVE: Excellent creditworthiness and stable finances support approval.<|im_end|>



### 4. Check label distribution before oversampling

In [7]:
import json

with open("/kaggle/working/train_formatted.jsonl", "r") as f:
    train_data = [json.loads(line) for line in f]

# Check raw train.jsonl for label counts (not the formatted chat version)
with open("/kaggle/input/datasets/ajawad06/loan-risk-data/train.jsonl", "r") as f:
    raw_train = [json.loads(line) for line in f]

from collections import Counter
print(Counter([e["label"] for e in raw_train]))

Counter({'refer': 129, 'decline': 104, 'approve': 91})


### 5. Oversample refer and decline examples

In [8]:
# New cell — oversample both refer and decline together

decline_examples = [e for e in raw_train if e["label"].lower() == "decline"]
refer_examples   = [e for e in raw_train if e["label"].lower() == "refer"]
approve_examples = [e for e in raw_train if e["label"].lower() == "approve"]

print(f"Before: approve={len(approve_examples)}, decline={len(decline_examples)}, refer={len(refer_examples)}")

# Keep refer oversampling from v2 (2x) AND add decline oversampling (2x)
oversampled_train = raw_train + refer_examples * 2 + decline_examples * 2

import random
random.seed(42)
random.shuffle(oversampled_train)

from collections import Counter
print(f"New total: {len(oversampled_train)}")
print(Counter([e["label"].lower() for e in oversampled_train]))

Before: approve=91, decline=104, refer=129
New total: 790
Counter({'refer': 387, 'decline': 312, 'approve': 91})


### 6. Reformat oversampled data into chat template (train_formatted_v2.jsonl)

In [9]:
def format_example(entry):
    profile = entry["profile"]
    label = entry["label"]
    reasoning = entry["reasoning"]
    user_msg = f"Classify the following loan application as approve, decline, or refer, and explain your reasoning.\n\nApplication: {profile}"
    assistant_msg = f"{label.upper()}: {reasoning}"
    text = f"<|im_start|>user\n{user_msg}<|im_end|>\n<|im_start|>assistant\n{assistant_msg}<|im_end|>"
    return {"text": text}

formatted_v2 = [format_example(e) for e in oversampled_train]

with open("/kaggle/working/train_formatted_v2.jsonl", "w") as f:
    for item in formatted_v2:
        f.write(json.dumps(item) + "\n")

print(f"Saved {len(formatted_v2)} examples")

Saved 790 examples


### 7. Train model: LoRA fine-tuning (5 epochs)

In [10]:
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import json

# --- Reload fresh base model (avoid stacking LoRA on top of LoRA) ---
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
)

# --- Load the oversampled, formatted data ---
with open("/kaggle/working/train_formatted_v2.jsonl", "r") as f:
    formatted_data_v2 = [json.loads(line) for line in f]

dataset_v2 = Dataset.from_list(formatted_data_v2)

# --- Add LoRA adapters ---
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# --- Training arguments (note: 5 epochs instead of 3) ---
training_args_v2 = TrainingArguments(
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    num_train_epochs=5,
    learning_rate=2e-4,
    fp16=True,
    bf16=False,
    logging_steps=5,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=42,
    output_dir="/kaggle/working/checkpoints_v2",
    save_strategy="no",
    average_tokens_across_devices=False,
)

# --- Trainer ---
trainer_v2 = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset_v2,
    dataset_text_field="text",
    max_seq_length=2048,
    args=training_args_v2,
)

# --- Train ---
trainer_stats_v2 = trainer_v2.train()

# --- Save the new adapter (separate folder, so you keep the original too) ---
model.save_pretrained("/kaggle/working/lora_adapter_v2")
tokenizer.save_pretrained("/kaggle/working/lora_adapter_v2")

==((====))==  Unsloth 2026.7.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth 2026.7.6 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/790 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 790 | Num Epochs = 5 | Total steps = 125
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,3.159702
10,2.782726
15,2.279861
20,1.987294
25,1.717733
30,1.647731
35,1.549780
40,1.499330
45,1.456663
50,1.367889


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/lora_adapter_v2/tokenizer_config.json.


('/kaggle/working/lora_adapter_v2/tokenizer_config.json',
 '/kaggle/working/lora_adapter_v2/chat_template.jinja',
 '/kaggle/working/lora_adapter_v2/tokenizer.json')

### 8. Run inference on held-out set

In [11]:
import json

FastLanguageModel.for_inference(model)

with open("/kaggle/input/datasets/ajawad06/loan-risk-data/held_out.jsonl", "r") as f:
    held_out = [json.loads(line) for line in f]

results_v2 = []

for entry in held_out:
    profile = entry["profile"]
    true_label = entry["label"]

    messages = [{"role": "user", "content": f"Classify the following loan application as approve, decline, or refer, and explain your reasoning.\n\nApplication: {profile}"}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.1,
        do_sample=False,
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    prediction = response.split("assistant")[-1].strip()

    results_v2.append({
        "profile": profile,
        "true_label": true_label,
        "prediction": prediction
    })

with open("/kaggle/working/inference_results_v2.jsonl", "w") as f:
    for r in results_v2:
        f.write(json.dumps(r) + "\n")

print(f"Ran inference on {len(results_v2)} held-out examples")

Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

Ran inference on 135 held-out examples


### 9. Evaluation: accuracy, precision/recall/F1, confusion matrix

In [12]:
import json
import re
from collections import Counter, defaultdict
from sklearn.metrics import classification_report, confusion_matrix

with open("/kaggle/working/inference_results_v2.jsonl", "r") as f:
    results_v2 = [json.loads(line) for line in f]

def extract_label(prediction):
    match = re.match(r"\s*(APPROVE|DECLINE|REFER)", prediction, re.IGNORECASE)
    return match.group(1).lower() if match else "unknown"

true_labels_v2 = []
pred_labels_v2 = []

for r in results_v2:
    pred = extract_label(r["prediction"])
    true = r["true_label"].lower()
    true_labels_v2.append(true)
    pred_labels_v2.append(pred)

correct_v2 = sum(t == p for t, p in zip(true_labels_v2, pred_labels_v2))
print(f"V2 Accuracy: {correct_v2/len(results_v2):.2%} ({correct_v2}/{len(results_v2)})")

print("\nV2 Classification Report:")
print(classification_report(true_labels_v2, pred_labels_v2, labels=["approve", "decline", "refer"], zero_division=0))

print("V2 Confusion matrix (rows=true, cols=predicted):")
print(confusion_matrix(true_labels_v2, pred_labels_v2, labels=["approve", "decline", "refer"]))

V2 Accuracy: 93.33% (126/135)

V2 Classification Report:
              precision    recall  f1-score   support

     approve       0.92      1.00      0.96        54
     decline       0.92      0.97      0.95        36
       refer       0.97      0.82      0.89        45

    accuracy                           0.93       135
   macro avg       0.94      0.93      0.93       135
weighted avg       0.94      0.93      0.93       135

V2 Confusion matrix (rows=true, cols=predicted):
[[54  0  0]
 [ 0 35  1]
 [ 5  3 37]]


### 10. Error analysis: refer cases the model missed

In [13]:
import json
with open("/kaggle/working/inference_results_v2.jsonl", "r") as f:
    results_v2 = [json.loads(line) for line in f]

for r in results_v2:
    if r["true_label"].lower() == "refer" and not r["prediction"].lower().startswith("refer"):
        print(f"True: refer | Predicted: {r['prediction'][:60]}...")
        print(f"Profile: {r['profile'][:150]}...\n")

True: refer | Predicted: DECLINE: Multiple low-paid gig income sources combined with ...
Profile: Applicant works multiple gig economy jobs earning approximately $47,000 annually. Credit score 626. Requests $18,000 for medical expenses. Existing de...

True: refer | Predicted: DECLINE: Seasonal income with concentrated reliance on one o...
Profile: Applicant works seasonally in construction, earning approximately $51,000 annually. Credit score 642. Requests a $17,000 loan for vehicle repairs and ...

True: refer | Predicted: APPROVE: Strong income offsetting modest credit profile risk...
Profile: Applicant recently transitioned from military service to a civilian logistics position paying $79,000 annually. Credit score is 708. Requests a $34,00...

True: refer | Predicted: APPROVE: Strong credit profile and substantial future earnin...
Profile: Applicant recently became a partner in a small accounting firm after working there for seven years. Annual income is expected to increase from 

### 11. Credit Analyst Agent: Core Building Blocks (Pre-LangChain)


In [17]:
"""
Credit Analyst Agent a) Core building blocks (pre-LangChain)

This is the layer BEFORE orchestration:
  1. Mock tools (DTI/DSCR/LTV calculator, mock credit bureau pull)
  2. Profile-text builder (turns structured applicant data + tool outputs
     into the same text format v6 was fine-tuned on)
  3. A function that calls v6 and parses its decision + reasoning

Wrapped in LangChain in the next cell: tools become @tool-decorated
functions, and the policy-rule gate + commercial-rule + branching logic
are added around v6's judgment step.
"""

import re
from dataclasses import dataclass, field
from typing import Optional, Literal


# 1. Applicant input

@dataclass
class Applicant:
    name: str
    monthly_income: float
    monthly_debt_payments: float          # existing debt obligations
    requested_loan_amount: float
    loan_purpose: str
    collateral_value: Optional[float]     # None for unsecured
    annual_net_operating_income: Optional[float] = None  # for DSCR (commercial)
    annual_debt_service: Optional[float] = None          # for DSCR
    employment_years: float = 0.0
    late_payments_last_2y: int = 0
    existing_defaults: int = 0


# 2. Mock tools

def calc_dti(applicant: Applicant) -> float:
    """Debt-to-Income ratio (%). Lower is better. Returns inf for commercial
    applicants (monthly_income == 0), where DTI doesn't apply — see DSCR."""
    if applicant.monthly_income <= 0:
        return float("inf")
    return round((applicant.monthly_debt_payments / applicant.monthly_income) * 100, 2)


def calc_dscr(applicant: Applicant) -> Optional[float]:
    """Debt Service Coverage Ratio — commercial loans only. >1.0 means income covers debt."""
    if not applicant.annual_net_operating_income or not applicant.annual_debt_service:
        return None
    if applicant.annual_debt_service <= 0:
        return None
    return round(applicant.annual_net_operating_income / applicant.annual_debt_service, 2)


def calc_ltv(applicant: Applicant) -> Optional[float]:
    """Loan-to-Value ratio (%) — only meaningful when collateral exists."""
    if not applicant.collateral_value or applicant.collateral_value <= 0:
        return None
    return round((applicant.requested_loan_amount / applicant.collateral_value) * 100, 2)


def mock_credit_bureau_pull(applicant: Applicant) -> dict:
    """Mock credit bureau API call (stands in for Equifax/TransUnion/etc.)."""
    base_score = 720
    base_score -= applicant.late_payments_last_2y * 15
    base_score -= applicant.existing_defaults * 100
    base_score += min(applicant.employment_years, 10) * 3
    score = max(300, min(850, round(base_score)))

    if score >= 740:
        band = "excellent"
    elif score >= 670:
        band = "good"
    elif score >= 580:
        band = "fair"
    else:
        band = "poor"

    return {
        "credit_score": score,
        "band": band,
        "late_payments_last_2y": applicant.late_payments_last_2y,
        "existing_defaults": applicant.existing_defaults,
        "open_tradelines": 3 + applicant.existing_defaults,
    }


# 3. Policy rule gate (hard-violation auto-decline, bypasses everything else)

def check_hard_policy_violations(applicant, bureau, dti):
    if bureau["existing_defaults"] >= 2:
        return "Auto-decline: 2+ existing defaults on file (hard policy rule)."
    if dti != float("inf") and dti > 65:
        return f"Auto-decline: DTI of {dti}% exceeds maximum allowable threshold (65%)."
    if bureau["credit_score"] < 500:
        return f"Auto-decline: credit score {bureau['credit_score']} below minimum floor (500)."
    return None


# 4. Profile-text builder — must match the text format v6 was trained on

def build_profile_text(applicant: Applicant, bureau: dict, dti: float,
                        dscr: Optional[float], ltv: Optional[float]) -> str:
    is_commercial = dscr is not None and applicant.monthly_income == 0

    lines = [f"Applicant: {applicant.name}"]

    if is_commercial:
        lines.append("Applicant type: commercial (evaluated via DSCR, not personal income)")
    else:
        lines.append(f"Annual income: ${applicant.monthly_income * 12:,.0f}")

    lines += [
        f"Requested loan amount: ${applicant.requested_loan_amount:,.0f}",
        f"Loan purpose: {applicant.loan_purpose}",
        (f"Debt-to-Income (DTI) ratio: {dti}%" if dti != float("inf")
         else "Debt-to-Income (DTI): N/A (commercial loan, evaluated via DSCR)"),
    ]
    if dscr is not None:
        lines.append(f"Debt Service Coverage Ratio (DSCR): {dscr}")
    if ltv is not None:
        lines.append(f"Loan-to-Value (LTV) ratio: {ltv}%")
    lines += [
        f"Credit score: {bureau['credit_score']} ({bureau['band']})",
        f"Late payments (last 2 years): {bureau['late_payments_last_2y']}",
        f"Existing defaults: {bureau['existing_defaults']}",
        f"Employment history: {applicant.employment_years} years",
    ]
    return "\n".join(lines)


# 5. finetuned model call

Decision = Literal["approve", "decline", "refer"]

@dataclass
class ModelVerdict:
    decision: Decision
    reasoning: str
    raw_output: str = field(repr=False, default="")


def build_chatml_prompt(profile_text: str) -> str:
    """Match this to the exact ChatML template used during v6 fine-tuning."""
    return (
        "<|im_start|>system\n"
        "You are a loan risk classifier. Given an applicant profile, respond with "
        "a decision (approve, decline, or refer) and a brief reasoning.\n"
        "<|im_end|>\n"
        f"<|im_start|>user\n{profile_text}\n<|im_end|>\n"
        "<|im_start|>assistant\n"
    )


def parse_decision(raw_output: str) -> Decision:
    match = re.search(r"\b(approve|decline|refer)\b", raw_output.lower())
    if match:
        return match.group(1)  # type: ignore
    return "refer"  # fail safe: unparseable output routes to human review


def call_v6(profile_text: str, model, tokenizer) -> ModelVerdict:
    """Calls the fine-tuned v6 checkpoint and parses its decision + reasoning."""
    prompt = build_chatml_prompt(profile_text)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output_ids = model.generate(**inputs, max_new_tokens=200, temperature=0.1)
    raw_output = tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    decision = parse_decision(raw_output)
    return ModelVerdict(decision=decision, reasoning=raw_output.strip(), raw_output=raw_output)

### 12. LangChain Orchestration Layer


In [21]:
"""
Credit Analyst Agent b) LangChain orchestration layer

Wraps the pre-LangChain building blocks as LangChain @tool components
and orchestrates them with an LCEL chain ending in a RunnableBranch
for the approve / decline / refer decision.

"""

from typing import Optional, Literal
from dataclasses import dataclass, asdict

from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain_core.runnables import RunnableLambda, RunnableBranch


# Pydantic schema mirroring Applicant, used as the tool input schema

class ApplicantSchema(BaseModel):
    name: str
    monthly_income: float
    monthly_debt_payments: float
    requested_loan_amount: float
    loan_purpose: str
    collateral_value: Optional[float] = None
    annual_net_operating_income: Optional[float] = None
    annual_debt_service: Optional[float] = None
    employment_years: float = 0.0
    late_payments_last_2y: int = 0
    existing_defaults: int = 0


def _to_applicant(a: ApplicantSchema) -> Applicant:
    return Applicant(**a.model_dump())


# Tools — one @tool per mock function, same interface a real integration

@tool("calc_dti_tool", args_schema=ApplicantSchema)
def calc_dti_tool(**kwargs) -> float:
    """Calculate the applicant's Debt-to-Income (DTI) ratio as a percentage."""
    return calc_dti(_to_applicant(ApplicantSchema(**kwargs)))


@tool("calc_dscr_tool", args_schema=ApplicantSchema)
def calc_dscr_tool(**kwargs) -> Optional[float]:
    """Calculate Debt Service Coverage Ratio (DSCR) for commercial loan applicants."""
    return calc_dscr(_to_applicant(ApplicantSchema(**kwargs)))


@tool("calc_ltv_tool", args_schema=ApplicantSchema)
def calc_ltv_tool(**kwargs) -> Optional[float]:
    """Calculate Loan-to-Value (LTV) ratio as a percentage when collateral exists."""
    return calc_ltv(_to_applicant(ApplicantSchema(**kwargs)))


@tool("credit_bureau_tool", args_schema=ApplicantSchema)
def credit_bureau_tool(**kwargs) -> dict:
    """Pull (mock) credit bureau data: score, band, late payments, existing defaults."""
    return mock_credit_bureau_pull(_to_applicant(ApplicantSchema(**kwargs)))


class PolicyGateInput(BaseModel):
    dti: float
    bureau: dict = Field(description="Output of credit_bureau_tool")


@tool("policy_gate_tool", args_schema=PolicyGateInput)
def policy_gate_tool(dti: float, bureau: dict) -> Optional[str]:
    """Check hard policy-violation rules (defaults, DTI ceiling, credit floor).
    Returns a decline reason string if violated, else None."""
    return check_hard_policy_violations(None, bureau, dti)


class ModelJudgmentInput(BaseModel):
    profile_text: str


@tool("model_judgment_tool", args_schema=ModelJudgmentInput)
def model_judgment_tool(profile_text: str) -> dict:
    """Call the fine-tuned v6 checkpoint on a profile and return decision + reasoning."""
    verdict = call_v6(profile_text, model=model, tokenizer=tokenizer)
    return {"decision": verdict.decision, "reasoning": verdict.reasoning}


class CommercialRuleInput(BaseModel):
    dscr: float
    credit_score: int
    existing_defaults: int
    late_payments_last_2y: int


@tool("commercial_rule_tool", args_schema=CommercialRuleInput)
def commercial_rule_tool(dscr: float, credit_score: int, existing_defaults: int,
                          late_payments_last_2y: int) -> dict:
    """Deterministic underwriting rule for commercial/DSCR-driven applicants,
    used in place of v6 (which was never fine-tuned on commercial profiles)."""
    if dscr < 1.0:
        return {"decision": "decline",
                "reasoning": f"DSCR of {dscr} is below 1.0 — income does not "
                              f"cover debt service. Deterministic commercial rule."}
    if dscr >= 1.5 and credit_score >= 700 and existing_defaults == 0 and late_payments_last_2y == 0:
        return {"decision": "approve",
                "reasoning": f"DSCR of {dscr}, credit score {credit_score}, "
                              f"clean payment history. Deterministic commercial rule."}
    return {"decision": "refer",
            "reasoning": f"DSCR of {dscr} and credit score {credit_score} don't "
                          f"meet the clear-approve or clear-decline thresholds — "
                          f"needs human underwriter judgment. Deterministic commercial rule."}


TOOLS = [calc_dti_tool, calc_dscr_tool, calc_ltv_tool, credit_bureau_tool,
         policy_gate_tool, model_judgment_tool, commercial_rule_tool]


# Structured credit memo

@dataclass
class CreditMemo:
    applicant_name: str
    decision: Literal["approve", "decline", "refer"]
    decision_source: Literal["policy_rule", "commercial_rule", "v6_model"]
    reasoning: str
    dti: float
    dscr: Optional[float]
    ltv: Optional[float]
    credit_score: int
    credit_band: str
    red_flags: list
    next_action: str

    def as_text(self) -> str:
        flags = "\n".join(f"  - {f}" for f in self.red_flags) if self.red_flags else "  - none"
        dscr_line = f"\n  DSCR: {self.dscr}" if self.dscr is not None else ""
        ltv_line = f"\n  LTV:  {self.ltv}%" if self.ltv is not None else ""
        dti_display = "N/A (commercial)" if self.dti == float("inf") else f"{self.dti}%"
        return (
            f"{'='*60}\n"
            f"CREDIT MEMO — {self.applicant_name}\n"
            f"{'='*60}\n"
            f"DECISION: {self.decision.upper()}  (source: {self.decision_source})\n\n"
            f"Ratios:\n"
            f"  DTI:  {dti_display}{dscr_line}{ltv_line}\n"
            f"  Credit score: {self.credit_score} ({self.credit_band})\n\n"
            f"Red flags:\n{flags}\n\n"
            f"Reasoning:\n  {self.reasoning}\n\n"
            f"Next action: {self.next_action}\n"
        )


def _red_flags(applicant: Applicant, bureau: dict, dti: float, ltv: Optional[float]) -> list:
    flags = []
    if bureau["existing_defaults"] > 0:
        flags.append(f"{bureau['existing_defaults']} existing default(s) on record")
    if bureau["late_payments_last_2y"] >= 3:
        flags.append(f"{bureau['late_payments_last_2y']} late payments in the last 2 years")
    if dti != float("inf") and dti > 45:
        flags.append(f"elevated DTI ({dti}%)")
    if ltv is not None and ltv > 90:
        flags.append(f"high LTV ({ltv}%) — thin collateral cushion")
    if applicant.employment_years < 1:
        flags.append("under 1 year employment history")
    return flags

# LCEL orchestration chain

def _intake(applicant: Applicant) -> dict:
    """Step 1: intake + step 2: run all tools."""
    a_dict = asdict(applicant)
    dti = calc_dti_tool.invoke(a_dict)
    dscr = calc_dscr_tool.invoke(a_dict)
    ltv = calc_ltv_tool.invoke(a_dict)
    bureau = credit_bureau_tool.invoke(a_dict)
    red_flags = _red_flags(applicant, bureau, dti, ltv)
    return {
        "applicant": applicant,
        "dti": dti, "dscr": dscr, "ltv": ltv,
        "bureau": bureau, "red_flags": red_flags,
    }


def _policy_and_judgment(state: dict) -> dict:
    """Step 3: policy gate. Step 3b: commercial rule (bypasses v6). Step 4: v6 judgment."""
    decline_reason = policy_gate_tool.invoke({"dti": state["dti"], "bureau": state["bureau"]})
    if decline_reason:
        state["decision"] = "decline"
        state["decision_source"] = "policy_rule"
        state["reasoning"] = decline_reason
        return state

    if state["applicant"].monthly_income == 0 and state["dscr"] is not None:
        verdict = commercial_rule_tool.invoke({
            "dscr": state["dscr"],
            "credit_score": state["bureau"]["credit_score"],
            "existing_defaults": state["bureau"]["existing_defaults"],
            "late_payments_last_2y": state["bureau"]["late_payments_last_2y"],
        })
        state["decision"] = verdict["decision"]
        state["decision_source"] = "commercial_rule"
        state["reasoning"] = verdict["reasoning"]
        return state

    profile_text = build_profile_text(state["applicant"], state["bureau"],
                                       state["dti"], state["dscr"], state["ltv"])
    verdict = model_judgment_tool.invoke({"profile_text": profile_text})
    state["decision"] = verdict["decision"]
    state["decision_source"] = "v6_model"
    state["reasoning"] = verdict["reasoning"]
    return state


def _handle_approve(state: dict) -> dict:
    state["next_action"] = "Auto-proceed: loan approved, forward to disbursement workflow."
    return state


def _handle_decline(state: dict) -> dict:
    state["next_action"] = "Auto-reject: applicant notified with reasoning; case closed."
    return state


def _handle_refer(state: dict) -> dict:
    state["next_action"] = "Escalated: routed to human underwriter for manual review."
    return state


_decision_branch = RunnableBranch(
    (lambda s: s["decision"] == "approve", RunnableLambda(_handle_approve)),
    (lambda s: s["decision"] == "decline", RunnableLambda(_handle_decline)),
    RunnableLambda(_handle_refer),  # default: refer
)


def _to_memo(state: dict) -> CreditMemo:
    return CreditMemo(
        applicant_name=state["applicant"].name,
        decision=state["decision"],
        decision_source=state["decision_source"],
        reasoning=state["reasoning"],
        dti=state["dti"], dscr=state["dscr"], ltv=state["ltv"],
        credit_score=state["bureau"]["credit_score"],
        credit_band=state["bureau"]["band"],
        red_flags=state["red_flags"],
        next_action=state["next_action"],
    )


credit_analyst_chain = (
    RunnableLambda(_intake)
    | RunnableLambda(_policy_and_judgment)
    | _decision_branch
    | RunnableLambda(_to_memo)
)


def review_applicant(applicant: Applicant) -> CreditMemo:
    """Public entry point: run the full 5-step agent on one applicant."""
    return credit_analyst_chain.invoke(applicant)

### 13. Demo: End-to-End Test Cases


In [24]:
demo_applicants = [
    Applicant(
        name="Tariq Textiles v2 (strong DSCR commercial, retest)",
        monthly_income=0, monthly_debt_payments=0,
        requested_loan_amount=5_000_000, loan_purpose="working capital",
        collateral_value=None,
        annual_net_operating_income=3_000_000, annual_debt_service=1_800_000,
        employment_years=12, late_payments_last_2y=0, existing_defaults=0,
    ),
    Applicant(
        name="Saima (retail, high but sub-ceiling DTI)",
        monthly_income=80_000, monthly_debt_payments=48_000,
        requested_loan_amount=300_000, loan_purpose="home renovation",
        collateral_value=350_000, employment_years=6,
        late_payments_last_2y=0, existing_defaults=0,
    ),
    Applicant(
        name="Waseem Traders (commercial, real policy violation)",
        monthly_income=0, monthly_debt_payments=0,
        requested_loan_amount=1_200_000, loan_purpose="inventory financing",
        collateral_value=900_000,
        annual_net_operating_income=600_000, annual_debt_service=500_000,
        employment_years=3, late_payments_last_2y=2, existing_defaults=2,
    ),
    Applicant(
        name="Zara Foods v2 (weak DSCR commercial, retest)",
        monthly_income=0, monthly_debt_payments=0,
        requested_loan_amount=2_500_000, loan_purpose="equipment financing",
        collateral_value=1_800_000,
        annual_net_operating_income=900_000, annual_debt_service=980_000,
        employment_years=4, late_payments_last_2y=1, existing_defaults=0,
    ),
    Applicant(
        name="Naveed (retail, real high DTI, should auto-decline)",
        monthly_income=50_000, monthly_debt_payments=40_000,
        requested_loan_amount=200_000, loan_purpose="personal",
        collateral_value=None, employment_years=3,
        late_payments_last_2y=1, existing_defaults=0,
    ),
]

for app in demo_applicants:
    memo = review_applicant(app)
    print(memo.as_text())

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


CREDIT MEMO — Tariq Textiles v2 (strong DSCR commercial, retest)
DECISION: APPROVE  (source: commercial_rule)

Ratios:
  DTI:  N/A (commercial)
  DSCR: 1.67
  Credit score: 750 (excellent)

Red flags:
  - none

Reasoning:
  DSCR of 1.67, credit score 750, clean payment history. Deterministic commercial rule.

Next action: Auto-proceed: loan approved, forward to disbursement workflow.

CREDIT MEMO — Saima (retail, high but sub-ceiling DTI)
DECISION: REFER  (source: v6_model)

Ratios:
  DTI:  60.0%
  LTV:  85.71%
  Credit score: 738 (good)

Red flags:
  - elevated DTI (60.0%)

Reasoning:
  REFER: High income is offset by a very high DTI relative to that income, meaning the requested payment capacity is weak even though overall creditworthiness is good; refer for underwriting judgment on affordability rather than default risk alone.

Next action: Escalated: routed to human underwriter for manual review.

CREDIT MEMO — Waseem Traders (commercial, real policy violation)
DECISION: DECLINE  (